In [1]:
import os, sys, json, hashlib, itertools, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image

REPO_URL = "https://github.com/Mugdha-Naik/deepfake-detector-project.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
sys.path.append(REPO_DIR)

from utils import get_dataset_source, load_ffpp_split_ids, ffpp_split_for_path

INPUT_ROOT = "/kaggle/input"

def find_one(name, roots):
    """Exactly one copy or an error (never silently use a stale output)."""
    for root in roots:
        matches = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f"{name}: {len(matches)} copies under {root}. Attach only one:\n" + "\n".join(matches))
    raise FileNotFoundError(f"{name} not found under {roots}")

def resolve_faces_root(root):
    """Find the attached 'step5_faces' folder (does not descend into it)."""
    found = []
    for d, dirnames, _ in os.walk(root):
        if "step5_faces" in dirnames:
            found.append(os.path.join(d, "step5_faces"))
            dirnames.remove("step5_faces")
    if len(found) != 1:
        raise RuntimeError(f"expected exactly one step5_faces folder under {root}, found: {found}")
    return found[0]

print("helpers ready")

Cloning into '/kaggle/working/repo'...


helpers ready


In [2]:
# ============================================================
# LOAD INPUTS
# ============================================================
faces_csv = find_one("quality_filtered_faces.csv", [INPUT_ROOT])
overall_csv = find_one("step5_overall_summary.csv", [INPUT_ROOT])
smap_csv = find_one("video_split_map.csv", [REPO_DIR, INPUT_ROOT])
FACES_ROOT = resolve_faces_root(INPUT_ROOT)

faces = pd.read_csv(faces_csv)
overall = pd.read_csv(overall_csv).iloc[0]
smap = pd.read_csv(smap_csv)

for label, p in [("faces", faces_csv), ("step5 summary", overall_csv), ("split map", smap_csv), ("faces root", FACES_ROOT)]:
    print(f"{label:14s} {p}")
print("\nface rows:", len(faces), "| videos:", faces["video_id"].nunique(), "| split-map videos:", len(smap))

needed = {"video_path", "label_name", "frame_number", "face_path", "x1", "y1", "x2", "y2", "confidence",
          "face_width", "face_height", "dataset", "video_id", "method", "split", "blur_score", "blur_pass", "quality_pass"}
assert needed <= set(faces.columns), f"not the scaled Step 5 output; missing: {needed - set(faces.columns)}"

faces          /kaggle/input/datasets/mugdhanaik/quality-faces-filtered/quality_filtered_faces.csv
step5 summary  /kaggle/input/datasets/mugdhanaik/step5-summary-overall/step5_overall_summary.csv
split map      /kaggle/input/datasets/mugdhanaik/video-maps/video_split_map.csv
faces root     /kaggle/input/notebooks/mugdhanaik/step5-regenerate/step5_faces

face rows: 28184 | videos: 951 | split-map videos: 13529


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 3:
        print(root, "| dirs:", dirs[:6], "| files:", files[:3])
    if depth >= 3:
        dirs[:] = []

/kaggle/input | dirs: ['notebooks', 'datasets'] | files: []
/kaggle/input/notebooks | dirs: ['mugdhanaik'] | files: []
/kaggle/input/notebooks/mugdhanaik | dirs: ['step5-regenerate'] | files: []
/kaggle/input/notebooks/mugdhanaik/step5-regenerate | dirs: ['step5_faces'] | files: ['__results__.html', '__notebook__.ipynb', '__output__.json']
/kaggle/input/datasets | dirs: ['mugdhanaik', 'xdxd003', 'reubensuju'] | files: []
/kaggle/input/datasets/mugdhanaik | dirs: ['video-maps', 'step5-summary-overall', 'quality-faces-filtered'] | files: []
/kaggle/input/datasets/mugdhanaik/video-maps | dirs: [] | files: ['video_split_map.csv']
/kaggle/input/datasets/mugdhanaik/step5-summary-overall | dirs: [] | files: ['step5_overall_summary.csv']
/kaggle/input/datasets/mugdhanaik/quality-faces-filtered | dirs: [] | files: ['quality_filtered_faces.csv']
/kaggle/input/datasets/xdxd003 | dirs: ['ff-c23'] | files: []
/kaggle/input/datasets/xdxd003/ff-c23 | dirs: ['FaceForensics++_C23'] | files: []
/kaggle/

In [4]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    hit = [d for d in dirs if d in ("step5_faces", "step5_quality")]
    if hit:
        print(root, "->", hit)
    dirs[:] = [d for d in dirs if d != "step5_faces"]   # don't walk into the crops
print("done")

/kaggle/input/notebooks/mugdhanaik/step5-regenerate -> ['step5_faces']
done


In [5]:
# ============================================================
# BUILD THE MANIFEST
# ============================================================
LABEL_TO_INT = {"REAL": 0, "FAKE": 1}          # FAKE is the positive class everywhere in Phase 2

m = faces.copy()

# path relative to the step5_faces folder, e.g. "faces/video_12_f30.png"  (session independent)
def to_rel(p):
    p = str(p).replace("\\", "/")
    assert "step5_faces/" in p, f"unexpected face path: {p}"
    return p.split("step5_faces/", 1)[1]

m["face_rel"] = m["face_path"].apply(to_rel)
m["video_key"] = m["video_path"].apply(lambda p: "/".join(str(p).replace("\\", "/").split("/")[-2:]))
m["face_id"] = m["video_id"].astype(str) + "_f" + m["frame_number"].astype(int).astype(str)
m["label"] = m["label_name"].map(LABEL_TO_INT)
m["condition"] = np.where(m["dataset"] == "FaceForensics++", "A_clean", "C_clean")
m["role"] = np.where(m["split"] == "test",
                     np.where(m["dataset"] == "FaceForensics++", "test_A", "test_C"), m["split"])
m["face_abs"] = m["face_rel"].apply(lambda r: os.path.join(FACES_ROOT, r))     # this session only; not saved

print(m.groupby(["role", "label_name"]).size().unstack(fill_value=0))

label_name  FAKE  REAL
role                  
test_A      3000  2883
test_C      4382  4381
train       5850  5891
val          897   900


In [6]:
# ============================================================
# INTEGRITY + LEAKAGE CHECKS  (all run; any failure stops the freeze)
# ============================================================
results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))

# --- identity / uniqueness ---
check("face_id is unique", m["face_id"].is_unique, f"{m['face_id'].duplicated().sum()} duplicates")
check("all faces passed the size filter", m["quality_pass"].astype(bool).all())
check("video_id <-> video_key is one-to-one",
      m["video_id"].nunique() == m["video_key"].nunique() == len(m[["video_id", "video_key"]].drop_duplicates()))

# --- files exist and open ---
missing = [p for p in m["face_abs"] if not os.path.exists(p)]
check("every face file exists", not missing, f"{len(missing)} missing; e.g. {missing[:2]}")
bad = 0
present_paths = [p for p in m["face_abs"] if os.path.exists(p)]
for p in random.Random(0).sample(present_paths, min(300, len(present_paths))):
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        bad += 1
check("random 300 face files open as images", bad == 0, f"{bad} unreadable")

# --- tags and labels ---
check("dataset tag agrees with the path", (m["video_path"].apply(get_dataset_source) == m["dataset"]).all())
REAL_FOLDERS = {"original", "Celeb-real", "YouTube-real"}
expected_label = np.where(m["method"].isin(REAL_FOLDERS), "REAL", "FAKE")
check("label agrees with the source folder (REAL folders: original / Celeb-real / YouTube-real)",
      (m["label_name"].values == expected_label).all(), f"{(m['label_name'].values != expected_label).sum()} mismatches")

# --- splits: one per video, consistent with the Step 6b map ---
check("no video appears in two splits", m.groupby("video_id")["split"].nunique().max() == 1)
j = m[["video_path", "split"]].drop_duplicates().merge(smap[["kaggle_path", "split"]], left_on="video_path",
                                                      right_on="kaggle_path", how="left", suffixes=("", "_map"))
check("every video is in the split map", j["split_map"].notna().all(), f"{j['split_map'].isna().sum()} videos not found")
check("split agrees with the Step 6b split map", (j["split"] == j["split_map"]).all(),
      f"{(j['split'] != j['split_map']).sum()} disagreements")

# --- FF++: re-derive the split from the OFFICIAL files, independent of the split map ---
SPLIT_DIR = os.path.join(REPO_DIR, "splits", "ffpp")
if not all(os.path.exists(os.path.join(SPLIT_DIR, f"{s}.json")) for s in ("train", "val", "test")):
    import urllib.request
    os.makedirs(SPLIT_DIR, exist_ok=True)
    for s in ("train", "val", "test"):
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{s}.json",
            os.path.join(SPLIT_DIR, f"{s}.json"))
id2split = load_ffpp_split_ids(SPLIT_DIR)
ff = m[m["dataset"] == "FaceForensics++"]
derived = ff["video_path"].apply(lambda p: ffpp_split_for_path(p, id2split))
check("FF++ split agrees with the official split files", (derived.values == ff["split"].values).all(),
      f"{(derived.values != ff['split'].values).sum()} faces disagree")

def ids_of(p):
    stem = os.path.splitext(os.path.basename(str(p).replace("\\", "/")))[0]
    return stem.split("_")
ids_by_split = {s: {i for p in g["video_path"].unique() for i in ids_of(p)} for s, g in ff.groupby("split")}
overlaps = {(a, b): len(ids_by_split[a] & ids_by_split[b]) for a, b in itertools.combinations(ids_by_split, 2)}
check("no FF++ identity (source or target id) crosses splits", all(v == 0 for v in overlaps.values()), str(overlaps))

# --- Celeb-DF is test-only and comes from the official test list ---
ce = m[m["dataset"] == "Celeb-DF"]
check("Celeb-DF faces appear only in the test split", set(ce["split"]) == {"test"})
official_test = set(smap.loc[(smap["dataset"] == "Celeb-DF") & (smap["split"] == "test"), "kaggle_path"])
check("every Celeb-DF video is in the official test list", set(ce["video_path"]) <= official_test)
check("no Celeb-DF video in train or val", not ((m["dataset"] == "Celeb-DF") & m["split"].isin(["train", "val"])).any())

# --- exact duplicate crops across splits (identical image bytes) ---
def md5_file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        h.update(f.read())
    return h.hexdigest()
present = m[m["face_abs"].apply(os.path.exists)].copy()
present["crop_md5"] = present["face_abs"].apply(md5_file)
shared = present.groupby("crop_md5")["split"].nunique()
check("no identical face crop appears in two different splits", (shared <= 1).all(), f"{int((shared > 1).sum())} shared crops; checked {len(present)} of {len(m)} files")

# --- group coverage ---
need_roles = {"train", "val", "test_A", "test_C"}
cov = m.groupby(["role", "label_name"])["video_id"].nunique().unstack(fill_value=0)
check("every role has both REAL and FAKE faces", set(cov.index) == need_roles and (cov.values > 0).all())
print("\nVideos per role x label:"); print(cov)
print("\nFaces per role x label:"); print(m.groupby(["role", "label_name"]).size().unstack(fill_value=0))
if (cov.values < 30).any():
    print("\nWARNING: some groups have fewer than 30 videos; confidence intervals will be wide.")

failed = [n for n, ok, _ in results if not ok]
print("\n" + "=" * 60)
print(f"{len(results) - len(failed)} of {len(results)} checks passed")
assert not failed, f"CHECKS FAILED, DO NOT FREEZE: {failed}"
print("ALL CHECKS PASSED: safe to freeze")

PASS  face_id is unique   [0 duplicates]
PASS  all faces passed the size filter
PASS  video_id <-> video_key is one-to-one
PASS  every face file exists   [0 missing; e.g. []]
PASS  random 300 face files open as images   [0 unreadable]
PASS  dataset tag agrees with the path
PASS  label agrees with the source folder (REAL folders: original / Celeb-real / YouTube-real)   [0 mismatches]
PASS  no video appears in two splits
PASS  every video is in the split map   [0 videos not found]
PASS  split agrees with the Step 6b split map   [0 disagreements]
PASS  FF++ split agrees with the official split files   [0 faces disagree]
PASS  no FF++ identity (source or target id) crosses splits   [{('test', 'train'): 0, ('test', 'val'): 0, ('train', 'val'): 0}]
PASS  Celeb-DF faces appear only in the test split
PASS  every Celeb-DF video is in the official test list
PASS  no Celeb-DF video in train or val
PASS  no identical face crop appears in two different splits   [0 shared crops; checked 28184 of 281

In [7]:
# ============================================================
# FREEZE: manifest + counts + metadata (hashes tie it to its inputs)
# ============================================================
OUT = "/kaggle/working/p2_manifest"
os.makedirs(OUT, exist_ok=True)

cols = ["face_id", "face_rel", "video_id", "video_key", "video_path", "dataset", "method", "label_name", "label",
        "split", "role", "condition", "frame_number", "x1", "y1", "x2", "y2", "confidence",
        "face_width", "face_height", "blur_score", "blur_pass"]
order = {"train": 0, "val": 1, "test_A": 2, "test_C": 3}
manifest = (m.assign(_o=m["role"].map(order))
              .sort_values(["_o", "dataset", "label_name", "video_id", "frame_number"])[cols]
              .reset_index(drop=True))
manifest_path = os.path.join(OUT, "faces_manifest.csv")
manifest.to_csv(manifest_path, index=False)

split_counts = (manifest.groupby(["role", "dataset", "label_name"])
                        .agg(videos=("video_id", "nunique"), faces=("face_id", "size")).reset_index())
split_counts.to_csv(os.path.join(OUT, "split_counts.csv"), index=False)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

tr = manifest[manifest["role"] == "train"]
n_real, n_fake = int((tr["label"] == 0).sum()), int((tr["label"] == 1).sum())
meta = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "manifest_sha256": sha256(manifest_path),
    "source_faces_csv_sha256": sha256(faces_csv),
    "n_faces": int(len(manifest)), "n_videos": int(manifest["video_id"].nunique()),
    "label_map": LABEL_TO_INT, "positive_class": "FAKE",
    "step5": {"max_faces_per_video": int(overall["max_faces_per_video"]),
              "frame_interval": int(overall["frame_interval"]), "seed": int(overall["seed"])},
    "train_class_weights": {"weight_REAL": round((n_real + n_fake) / (2 * n_real), 4),
                            "weight_FAKE": round((n_real + n_fake) / (2 * n_fake), 4)},
    "face_rel_note": "face_rel is relative to the step5_faces folder; resolve it with resolve_faces_root() in each notebook",
    "checks_passed": len(results),
}
with open(os.path.join(OUT, "manifest_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Saved to", OUT)
for n in ("faces_manifest.csv", "split_counts.csv", "manifest_meta.json"):
    print(" ", n)
print("\nmanifest sha256:", meta["manifest_sha256"])
print(split_counts.to_string(index=False))

Saved to /kaggle/working/p2_manifest
  faces_manifest.csv
  split_counts.csv
  manifest_meta.json

manifest sha256: 80eae250d797355a0e27180c7afa5753b642ce82554bf58bbf9a8391f95218f6
  role         dataset label_name  videos  faces
test_A FaceForensics++       FAKE     100   3000
test_A FaceForensics++       REAL      98   2883
test_C        Celeb-DF       FAKE     150   4382
test_C        Celeb-DF       REAL     150   4381
 train FaceForensics++       FAKE     195   5850
 train FaceForensics++       REAL     198   5891
   val FaceForensics++       FAKE      30    897
   val FaceForensics++       REAL      30    900
